#### beads-id: yaralpho-hgo
#### beads-status: synced

# $TITLE$: # Persistent Scheduler & Repository-Aware Processing Implementation Plan

**Goal:** Refactor the service to a Mongo-backed, repository-aware scheduler that processes batch items sequentially per batch, across multiple runtime-configurable agents, with retries, pause/resume, and a graceful restart endpoint for CI.

**Architecture:** A single web process (only one scheduler instance) drives a 10s periodic goroutine that reads Mongo for pending batches. It claims the next pending item of each eligible batch, assigns the first idle agent, and executes work via the existing worker interface. Batches are tied to repositories; agents are runtime entries. The legacy in-memory queue and epic concept are removed. Restart drains active work and optionally blocks the caller.

**Tech Stack:** Go 1.25, MongoDB driver already in use, Gorilla mux, existing logging/notify/tracker wiring. No new dependencies.

**Assumptions & Constraints:**
- Exactly one web process; many agents (workers) allowed.
- No per-batch concurrency: items within a batch run strictly sequentially; multiple batches may run in parallel if multiple agents are idle.
- Repository CRUD stores name/path; path validation required; unique name/path; deletion only when no active batches.
- Agent CRUD manages runtime workers (codex|copilot); status is busy|idle; default idle; deletion/update forbidden when busy.
- Batch is created pending; first failed item marks entire batch failed. Only one failed item is tracked; human fixes and restarts.
- Retry policy: per-item attempts up to `maxRetries`; after exhaustion, batch is failed. Batch restart endpoint resets failed item+batch to pending.
- Pause semantics: pause prevents new scheduling; in-progress item finishes; resume returns batch to pending.
- Restart endpoint: `/restart?wait` optional bool. If wait=true, HTTP blocks until all active work drains; used by CI to `curl ... && start`.
- Status sets: Batch {pending, in-progress, failed, paused, done}; Item {pending, in-progress, done, failed}; Agent {idle, busy}.
- Runs are per-item attempts (no epics). `/runs/{id}`, `/runs/{id}/events`, `/runs/{id}/events/live` remain; global `/runs` list is removed.
- Scheduler must skip paused or draining states; no new work when draining.
- First idle agent selection is simple (no balancing).
- No special filesystem security beyond basic path validation.

**API Surface (final form):**
- Repository CRUD: `POST/GET /repository`, `GET/PUT/DELETE /repository/{id}` (delete only if no active batches).
- Add batch: `POST /repository/{repoid}/add` (replaces old `/add`); creates pending batch, items pending, repository_id set.
- List batches: `GET /repository/{repoid}/batches?status=` (optional filter pending|in-progress|failed|paused|done).
- Pause/Resume: `PUT /repository/{repoid}/batch/{batchid}/pause`, `.../resume`.
- Restart failed batch: `PUT /repository/{repoid}/batch/{batchid}/restart` (only when batch failed; resets failed item to pending, attempts=0).
- List runs for batch: `GET /repository/{repoid}/batch/{batchid}/runs`.
- Run detail/events: `/runs/{id}`, `/runs/{id}/events`, `/runs/{id}/events/live` (unchanged).
- Agents CRUD: `POST/GET /agent`, `GET/PUT/DELETE /agent/{id}` (no delete/update when busy).
- System: `GET /version`; `POST/GET /restart?wait` (drain + optional blocking).

**Scheduler / Execution Rules:**
- Tick every 10s (configurable): if draining, skip; otherwise: find next pending batch with a pending item; ensure batch not paused; ensure no item in-progress in that batch (sequential rule); pick first idle agent; set agent busy; mark item in-progress; create Run with repository_id; execute.
- On success: item done; if all items done -> batch done; agent idle.
- On failure: increment item attempt; if attempts < maxRetries -> set item pending, batch pending; else set item failed + batch failed; agent idle.
- Pause/resume enforced in selection; pause never cancels in-progress item.
- Draining: Stop starting new work; wait for active runs to finish; `wait=true` blocks caller until drain complete; `wait=false` returns immediately while draining continues.

**Retry & Restart Semantics:**
- Item retries handled automatically up to maxRetries per item.
- Batch restart endpoint only allowed on failed batch; finds failed item, resets to pending/attempts=0, batch status pending; scheduler picks it up in next tick.

**Testing Expectations:**
- Unit tests for scheduler selection, sequential constraint, pause, retry, failure paths, agent busy/idle transitions.
- Integration tests with mocked agent executor and shortened tick interval: pending→in-progress→done; retry to failure; pause/resume; batch restart.
- Handler tests for all endpoints, validation, and forbidden operations (delete busy agent, delete repo with active batch, restart non-failed batch).
- Migration tests for new models/fields.


#### beads-id: yaralpho-hgo.1
#### beads-status: synced

# $TITLE$:  Task 1: Capture current models/handlers context

**Files:** Read `internal/storage/models.go`, `internal/storage/interfaces.go`, `internal/app/*.go`, `internal/queue/*`, `internal/consumer/*` (no edits).  
**Purpose:** Build a precise map of current Batch/Run shapes, queue usage, handler routes (including `/runs` endpoints), and where epics and in-memory queue are referenced so later edits don’t miss any code paths. Produce notes that enumerate every field and route needing change.  
**Outcome:** A short notes block (kept locally or in commit message scratch) listing existing models, queue wiring points, and handlers to touch/remove.  
**How to Verify:** Confirm notes include: Batch/Run fields, queue interfaces, consumer flow, all routes referencing runs/add/epic. No code diffs yet (`git status` clean).  
**Acceptance Criteria:** Notes are complete enough to guide edits without re-reading code; no files modified.


#### beads-id: yaralpho-hgo.2
#### beads-status: synced

# $TITLE$: Task 2: Define new domain models & constants

Files: internal/storage/models.go; internal/storage/interfaces.go; internal/storage/status.go (or equivalent). Purpose: Introduce Repository and Agent structs; extend Batch with repository_id and items []BatchItem{input,status,attempts}; extend Run with repository_id; define enums for BatchStatus, ItemStatus, AgentStatus with doc comments matching lifecycle rules. Outcome: Compilable models/interfaces reflecting repository-aware batches, item-level status/attempts, agent runtime model, and run linkage. How to Verify: go vet ./... and go test ./internal/storage/... (models compile); adjust stubs if needed. Acceptance Criteria: Public types documented; statuses match defined sets; no epic references remain in models.

#### beads-id: yaralpho-hgo.3
#### beads-status: synced

# $TITLE$: Task 3: Persistence layer updates (Mongo)

Files: internal/storage/mongo/*; internal/storage/mongo/*_test.go. Purpose: Implement Mongo persistence for Repository (unique name/path), Agent (unique name, runtime), Batch with items sub-doc and repository_id, Run with repository_id; add indexes in init; map new fields and enforce sequential item order. Outcome: Working CRUD for repositories/agents; batches store item status/attempts; runs store repository_id; indexes created. How to Verify: go test ./internal/storage/mongo/...; go vet; optional round-trip creating repo, agent, batch with items, run. Acceptance Criteria: Tests pass; uniqueness enforced; serialization round-trips item status/attempts; migrations/backfill handled or noted N/A.

#### beads-id: yaralpho-hgo.4
#### beads-status: synced

# $TITLE$: Task 4: Remove in-memory queue usage

Files: internal/queue/*; internal/app/app.go (wiring); internal/consumer/*; related tests. Purpose: Eliminate legacy in-memory producer/consumer so only scheduler drives work; remove queue wiring and dead paths. Outcome: Build succeeds without queue dependency; consumer no longer listens to queue. How to Verify: rg 'queue' internal | grep -v vendor (only intended refs); go test ./... builds consumer without queue. Acceptance Criteria: No runtime queue wiring remains; consumer entrypoint uses scheduler hook instead.

#### beads-id: yaralpho-hgo.5
#### beads-status: synced

# $TITLE$: Task 5: Scheduler interface skeleton

Files: internal/scheduler/interface.go; internal/scheduler/scheduler.go; internal/scheduler/scheduler_test.go. Purpose: Define Start/Stop/Tick API plus constructor parameters (interval, drain flag, storage + worker interfaces); document concurrency/drain expectations; stub implementation so downstream compiles. Outcome: Interface with comments and stub Scheduler implementing methods (no logic yet); compile-time tests exist. How to Verify: go test ./internal/scheduler -run TestSchedulerInterface; go vet ./internal/scheduler. Acceptance Criteria: Tick preconditions documented (single process, no new work when draining); code compiles.

#### beads-id: yaralpho-hgo.6
#### beads-status: synced

# $TITLE$: Task 6: Implement scheduler tick selection

Files: internal/scheduler/scheduler.go; internal/scheduler/scheduler_test.go. Purpose: Implement Tick: skip if draining; fetch next pending batch with pending item; enforce no in-progress item in that batch (sequential); skip paused; pick first idle agent; atomically set agent busy, item in-progress, create Run; dispatch execution via worker callback. Outcome: Functional Tick with unit tests for no batches, no agents, paused, happy path, sequential enforcement. How to Verify: go test ./internal/scheduler -run TestTick*; inspect logs if using test logger spy. Acceptance Criteria: Agent busy set before execution; item status persisted in-progress; batch status in-progress; sequential rule enforced.

#### beads-id: yaralpho-hgo.7
#### beads-status: synced

# $TITLE$: Task 7: Worker execution + retry handling

Files: internal/scheduler/scheduler.go; internal/scheduler/scheduler_test.go; internal/consumer/worker.go (or new adapter). Purpose: Handle execution completion: success -> item done; if last item then batch done; failure -> increment attempts, if attempts < maxRetries set item pending and batch pending, else mark item failed and batch failed; agent always set idle; create Run per attempt with repository_id. Outcome: Retry-aware execution flow with tests for success, retry-then-success, retry-exhausted leading to batch failed. How to Verify: go test ./internal/scheduler -run TestRetry*; assert agent returns idle in each branch. Acceptance Criteria: Batch fails immediately when an item exhausts retries; only one failed item tracked; runs record attempts.

#### beads-id: yaralpho-hgo.8
#### beads-status: synced

# $TITLE$: Task 8: Pause/resume behavior enforcement

Files: internal/scheduler/scheduler.go; internal/scheduler/scheduler_test.go. Purpose: Ensure paused batches are never selected; in-progress item may finish; resume moves paused batch back to pending (if not failed/done). Outcome: Pause flag checked in selection; resume updates status appropriately; tests cover paused skip and resume pickup. How to Verify: go test ./internal/scheduler -run TestPause*. Acceptance Criteria: No item starts while batch paused; after resume, next tick schedules pending item.

#### beads-id: yaralpho-hgo.9
#### beads-status: synced

# $TITLE$: Task 9: Graceful drain/restart plumbing

Files: internal/scheduler/scheduler.go; internal/app/restart_handler.go; internal/app/app.go; internal/app/restart_handler_test.go. Purpose: Add draining state and active-run tracking; Stop prevents new ticks; restart handler sets draining and blocks if wait=true or returns immediately if false; scheduler rejects new work during drain. Outcome: Functional /restart?wait supporting CI curl-wait pattern; drain finishes active items then stops. How to Verify: go test ./internal/app -run TestRestart* with fake scheduler; manual curl optional. Acceptance Criteria: No new work starts after drain begins; wait=true blocks until active=0; appropriate status codes (e.g., 202 non-wait, 200/204 when drained).

#### beads-id: yaralpho-hgo.10
#### beads-status: synced

# $TITLE$: Task 10: Repository CRUD endpoints

Files: internal/app/repository_handlers.go; internal/app/app.go; internal/app/repository_handlers_test.go. Purpose: Implement POST/GET list/GET by id/PUT/DELETE with validation (unique name/path, path format) and forbid delete when any active/pending/in-progress batches exist. Outcome: Repository API fully functional with proper errors. How to Verify: go test ./internal/app -run TestRepository*; optional curl manual checks. Acceptance Criteria: 409 on duplicate name/path; 400 on bad input; 409 on delete with active batches; responses include repo_id and timestamps.

#### beads-id: yaralpho-hgo.11
#### beads-status: synced

# $TITLE$: Task 11: Agent CRUD endpoints

Files: internal/app/agent_handlers.go; internal/app/app.go; internal/app/agent_handlers_test.go. Purpose: Manage runtime agents (codex|copilot) with POST/GET list/GET/PUT/DELETE; default idle; block delete/update when busy; validate agent type. Outcome: Agent API fully functional with protections. How to Verify: go test ./internal/app -run TestAgent*. Acceptance Criteria: Invalid agent type rejected; busy agent cannot be deleted or status-updated; new agents start idle.

#### beads-id: yaralpho-hgo.12
#### beads-status: synced

# $TITLE$: Task 12: Batch creation under repository

Files: internal/app/batch_handlers.go (existing add handler modification); internal/app/app.go; internal/app/batch_handlers_test.go. Purpose: Replace old /add with POST /repository/{repoid}/add; create batch with repository_id, status pending; initialize items list from request with status pending and attempts=0; no queue enqueuing. Outcome: New endpoint returns batch id/status; old /add removed. How to Verify: go test ./internal/app -run TestBatchCreate*; rg '/add' shows only new route references. Acceptance Criteria: Batch persisted pending with items pending; no queue interaction; 404 when repository not found.

#### beads-id: yaralpho-hgo.13
#### beads-status: synced

# $TITLE$: Task 13: List batches by repository with status filter

Files: internal/app/batch_handlers.go; internal/app/batch_handlers_test.go. Purpose: Implement GET /repository/{repoid}/batches?status= to return batches for repo with optional status filter (pending|in-progress|failed|paused|done), 400 on unknown status. Outcome: Filtered or full list endpoint. How to Verify: go test ./internal/app -run TestBatchList*. Acceptance Criteria: Correct filtering; empty list when none; pagination consistent with existing style if any.

#### beads-id: yaralpho-hgo.14
#### beads-status: synced

# $TITLE$: Task 14: Pause/resume endpoints

Files: internal/app/batch_handlers.go; internal/app/batch_handlers_test.go. Purpose: Add PUT /repository/{repoid}/batch/{batchid}/pause and /resume; pause allowed unless batch done/failed; resume only from paused; scheduler must respect flag. Outcome: Status transitions persisted and exposed. How to Verify: go test ./internal/app -run TestBatchPauseResume*. Acceptance Criteria: Pause does not alter in-progress item; resume sets pending; invalid transitions return 409.

#### beads-id: yaralpho-hgo.15
#### beads-status: synced

# $TITLE$: Task 15: Restart failed batch endpoint

Files: internal/app/batch_handlers.go; internal/app/batch_handlers_test.go. Purpose: Implement PUT /repository/{repoid}/batch/{batchid}/restart only when batch failed; find failed item, reset to pending and attempts=0; set batch status pending. Outcome: Failed batches can be retried after fixes. How to Verify: go test ./internal/app -run TestBatchRestart*. Acceptance Criteria: 409 if batch not failed; after restart scheduler picks up next tick; response returns updated batch state.

#### beads-id: yaralpho-hgo.16
#### beads-status: synced

# $TITLE$: Task 16: Runs listing scoped to batch

Files: internal/app/run_handlers.go; internal/app/run_handlers_test.go. Purpose: Implement GET /repository/{repoid}/batch/{batchid}/runs to list runs for a batch; remove global GET /runs; ensure repository scoping. Outcome: Batch-scoped run listing replaces legacy global list. How to Verify: go test ./internal/app -run TestBatchRunsList*; rg 'HandleFunc(/runs' shows only detail/events routes remain. Acceptance Criteria: Returns runs belonging to batch; 404 on unknown batch; old list endpoint removed.

#### beads-id: yaralpho-hgo.17
#### beads-status: synced

# $TITLE$: Task 17: Run detail/event handlers alignment

Files: internal/app/run_handlers.go; internal/app/run_handlers_test.go. Purpose: Keep /runs/{id}, /runs/{id}/events, /runs/{id}/events/live working with new schema (repository_id present, no epic); ensure storage calls use updated structs. Outcome: Run detail/event routes function unchanged externally but on new model. How to Verify: go test ./internal/app -run TestRunDetail*. Acceptance Criteria: Responses include repository context where applicable; no epic references; handlers compile.

#### beads-id: yaralpho-hgo.18
#### beads-status: synced

# $TITLE$: Task 18: Remove epic concept entirely

Files: internal/**/* where epic logic exists; tracker usage; related tests. Purpose: Remove branching on epic vs task; align terminology to single-task batches; adjust tracker as needed. Outcome: No code paths or tests refer to epics; tracker simplified/stubbed accordingly. How to Verify: rg 'epic' internal yields only explanatory comments; go test ./... passes. Acceptance Criteria: Build green; functionality matches single-task batches.

#### beads-id: yaralpho-hgo.19
#### beads-status: synced

# $TITLE$: Task 19: Version endpoint

Files: internal/app/system_handlers.go; internal/app/app.go; internal/app/system_handlers_test.go. Purpose: Expose GET /version returning git commit SHA or 'dev' fallback. Outcome: Endpoint returns JSON version string. How to Verify: go test ./internal/app -run TestVersion*; optional curl. Acceptance Criteria: 200 with version field; no side effects.

#### beads-id: yaralpho-hgo.20
#### beads-status: synced

# $TITLE$: Task 20: Restart endpoint wiring

Files: internal/app/app.go; internal/app/restart_handler.go. Purpose: Register /restart route with optional wait bool param and connect to drain logic; ensure handler reachable via router. Outcome: Endpoint reachable and functions per drain implementation. How to Verify: go test ./internal/app -run TestRestart*; optional curl -i '/restart?wait=true'. Acceptance Criteria: Correct status codes; respects wait parameter; no duplicate routes.

#### beads-id: yaralpho-hgo.21
#### beads-status: synced

# $TITLE$: Task 21: Observability & logging

Files: internal/scheduler/scheduler.go; handlers for restart/batch transitions. Purpose: Add structured logs for scheduler decisions (no batches, no agents, paused, claim, retry, fail), drain start/stop, pause/resume actions, agent busy/idle transitions; include batch_id, item index, agent_id, repository_id, attempt counts; avoid sensitive data. Outcome: Traceable logs for scheduling and restart behavior. How to Verify: Unit tests with logger spy/assert; manual run shows logs on claim/retry/fail. Acceptance Criteria: Every branch logs entry/exit/errors with identifiers; no secrets logged.

#### beads-id: yaralpho-hgo.22
#### beads-status: synced

# $TITLE$: Task 22: Configuration defaults

Files: internal/config/config.go (or equivalent); internal/app/app.go wiring; docs. Purpose: Add config fields for scheduler interval (default 10s), maxRetries, restart wait timeout; bind from env YARALPHO_*; document defaults; inject into scheduler. Outcome: Configurable behavior with safe defaults; scheduler uses injected values. How to Verify: go test ./internal/config/...; unit test for defaults when env unset; integration overrides interval for fast tests. Acceptance Criteria: Missing env uses defaults; invalid values handled gracefully (error or fallback).

#### beads-id: yaralpho-hgo.23
#### beads-status: synced

# $TITLE$: Task 23: Integration tests (happy, retry, pause/resume, batch restart)

Files: internal/app/integration_test.go or test/integration/*; fakes for agent execution. Purpose: Add deterministic end-to-end tests covering: (1) repo+task -> batch pending; (2) repo+worker+task -> pending->in-progress->done, agent idle; (3) failing task retried until maxRetries then batch failed, agent idle; (4) paused batch not picked up until resume then completes. Use shortened tick interval via config override. Outcome: Reliable integration suite validating scheduler and API behavior. How to Verify: go test ./... -run TestIntegration* with fast runtime. Acceptance Criteria: All scenarios pass consistently; uses mocked agent executor; no long sleeps.

#### beads-id: yaralpho-hgo.24
#### beads-status: synced

# $TITLE$: Task 24: Docs update

Files: README.md; docs/design/scheduler.md or this plan file; other relevant docs. Purpose: Document new APIs, status lifecycles, retry rules, pause/resume, restart semantics, config envs, sequential batch rule; note removal of /runs list and epic concept. Outcome: Developer docs accurate to implementation with quickstart examples. How to Verify: Manual review; router vs docs cross-check; rg '/runs' in docs shows only correct references. Acceptance Criteria: All new endpoints documented; old ones removed; restart usage with wait param explained for CI.

#### beads-id: yaralpho-hgo.25
#### beads-status: synced

# $TITLE$: Task 25: Cleanup dead code

Files: internal/queue/* remnants; obsolete tests; go.mod/go.sum if needed. Purpose: Remove unused artifacts post-refactor to reduce maintenance and confusion; tidy modules. Outcome: Dead code removed; dependencies clean. How to Verify: rg 'queue' internal minimal; go mod tidy; go test ./.... Acceptance Criteria: Build green; no unused packages; repo clean.

#### beads-id: yaralpho-hgo.26
#### beads-status: synced

# $TITLE$: Task 26: Final verification & push

Files: commands only. Purpose: Run full quality gates and sync per AGENTS workflow: go test ./..., go vet ./..., git pull --rebase, bd sync, git push; ensure clean worktree. Outcome: All tests/linters green; changes pushed. How to Verify: Capture command outputs; git status shows up to date with origin. Acceptance Criteria: Tests pass; push succeeds; worktree clean.